<a href="https://colab.research.google.com/github/1stzl01/Reserving/blob/main/Mack_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Measuring the Variability of Chain Ladder Reserve Estimates
### Based on the paper by Thomas Mack

This notebook illustrates Thomas Mack's "distribution-free" approach to quantifying the standard error (variability) of chain ladder reserve estimates. While the chain ladder method is popular for its simplicity, it implicitly relies on three strong statistical assumptions:
1. **Expected Value:** $E(C_{i,k+1}|C_{i1}, ..., C_{ik}) = C_{ik}f_k$ (The expected claims in the next development period depend only on the current total and a uniform development factor $f_k$).
2. **Independence:** Accident years are strictly independent.
3. **Variance:** $Var(C_{i,k+1}|C_{i1}, ..., C_{ik}) = C_{ik}\alpha_k^2$ (The variance of the next development period is proportional to the accumulated claims amount to date).

This notebook generates a synthetic run-off triangle, applies Mack's standard error formulas, and demonstrates how to build confidence intervals for the reserves. It also includes an automated diagnostic test to check for calendar-year distortions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# ==========================================
# 1. LOAD DATA (MACK 10x10 EXAMPLE)
# ==========================================
I = 10 # Number of accident years and development years

# Cumulative incurred case losses in $1000 (Automatic Facultative General Liability)
# Using tuples (parentheses) to prevent chat formatting bugs!
C = np.array((
    (5012,  8269, 10907, 11805, 13539, 16181, 18009, 18608, 18662, 18834),
    ( 106,  4285,  5396, 10666, 13782, 15599, 15496, 16169, 16704, np.nan),
    (3410,  8992, 13873, 16141, 18735, 22214, 22863, 23466, np.nan, np.nan),
    (5655, 11555, 15766, 21266, 23425, 26083, 27067, np.nan, np.nan, np.nan),
    (1092,  9565, 15836, 22169, 25955, 26180, np.nan, np.nan, np.nan, np.nan),
    (1513,  6445, 11702, 12935, 15852, np.nan, np.nan, np.nan, np.nan, np.nan),
    ( 557,  4020, 10946, 12314, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan),
    (1351,  6947, 13112, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan),
    (3133,  5395, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan),
    (2063, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)
))

triangle_df = pd.DataFrame(C,
                           index=[f"Acc_Yr_{i+1}" for i in range(I)],
                           columns=[f"Dev_Yr_{k+1}" for k in range(I)])

print("Mack's Original Cumulative Claims Triangle:")
display(triangle_df)

Mack's Original Cumulative Claims Triangle:


,Dev_Yr_1,Dev_Yr_2,Dev_Yr_3,Dev_Yr_4,Dev_Yr_5,Dev_Yr_6,Dev_Yr_7,Dev_Yr_8,Dev_Yr_9,Dev_Yr_10
Acc_Yr_1,5012.0,8269.0,10907.0,11805.0,13539.0,16181.0,18009.0,18608.0,18662.0,18834.0
Acc_Yr_2,106.0,4285.0,5396.0,10666.0,13782.0,15599.0,15496.0,16169.0,16704.0,NaN
Acc_Yr_3,3410.0,8992.0,13873.0,16141.0,18735.0,22214.0,22863.0,23466.0,NaN,NaN
Acc_Yr_4,5655.0,11555.0,15766.0,21266.0,23425.0,26083.0,27067.0,NaN,NaN,NaN
Acc_Yr_5,1092.0,9565.0,15836.0,22169.0,25955.0,26180.0,NaN,NaN,NaN,NaN
Acc_Yr_6,1513.0,6445.0,11702.0,12935.0,15852.0,NaN,NaN,NaN,NaN,NaN
Acc_Yr_7,557.0,4020.0,10946.0,12314.0,NaN,NaN,NaN,NaN,NaN,NaN
Acc_Yr_8,1351.0,6947.0,13112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Acc_Yr_9,3133.0,5395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Acc_Yr_10,2063.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 1: Calculate Standard Age-to-Age Factors ($f_k$)
The traditional chain ladder method estimates the development factor $f_k$ as the volume-weighted average of the observed individual development factors.
Formula: $\hat{f}_k = \frac{\sum_{j=1}^{I-k} C_{j,k+1}}{\sum_{j=1}^{I-k} C_{j,k}}$

In [ ]:
# ==========================================
# 2. CALCULATE DEVELOPMENT FACTORS & PROJECT ULTIMATE CLAIMS
# ==========================================

f_k = np.zeros(I - 1)

# Calculate f_k for each development column
for k in range(I - 1):
    # Sum all available claims in period k+1 and divide by sum of claims in period k
    # We only sum up to row I-k-1 (the last row with data in column k+1)
    sum_num = np.nansum(C[:I-k-1, k+1])
    sum_den = np.nansum(C[:I-k-1, k])
    f_k[k] = sum_num / sum_den

print("Estimated Development Factors (f_k):")
print(np.round(f_k, 3))

# Complete the lower half of the triangle to project ultimate claims
C_full = np.copy(C)
for i in range(1, I):
    for k in range(I - i, I):
        C_full[i, k] = C_full[i, k-1] * f_k[k-1]

# Calculate Individual Reserves (Ultimate - Latest Known)
latest_known = np.array([C[i, I-i-1] for i in range(I)])
ultimate_claims = C_full[:, I-1]
reserves = ultimate_claims - latest_known
total_reserve = np.sum(reserves)

print("\nProjected Outstanding Reserves by Accident Year:")
print(np.round(reserves, 0))
print(f"\nOverall Estimated Reserve: {total_reserve:,.0f}")

Estimated Development Factors (f_k):
[2.999 1.624 1.271 1.172 1.113 1.042 1.033 1.017 1.009]

Projected Outstanding Reserves by Accident Year:
[    0.   154.   617.  1636.  2747.  3649.  5435. 10907. 10650. 16339.]

Overall Estimated Reserve: 52,135


### Step 2: Calculate Variance Parameters ($\alpha_k^2$)
Mack proves that the volume-weighted chain ladder implies the variance of the next period is proportional to the current accumulated claims.
We estimate the proportionality constant $\alpha_k^2$ using the unbiased estimator:
$\alpha_k^2 = \frac{1}{I-k-1} \sum_{i=1}^{I-k} C_{i,k} \left( \frac{C_{i,k+1}}{C_{i,k}} - f_k \right)^2$

Because the formula cannot estimate the parameter for the final development period ($\alpha_{I-1}^2$) as there is only one observation, Mack suggests extrapolating it. We will extrapolate it using $\alpha_{I-1}^2 = \min(\frac{\alpha_{I-2}^4}{\alpha_{I-3}^2}, \alpha_{I-2}^2)$.


In [ ]:
# ==========================================
# 3. CALCULATE VARIANCE PARAMETERS (alpha_k^2)
# ==========================================

alpha_2 = np.zeros(I - 1)

# Calculate alpha_2 for k = 0 to I-3
for k in range(I - 2):
    sum_var = 0
    # Loop over all rows that have data for BOTH k and k+1
    for i in range(I - k - 1):
        individual_factor = C[i, k+1] / C[i, k]
        sum_var += C[i, k] * (individual_factor - f_k[k])**2

    # Divide by the degrees of freedom (I - k - 1 in Python's 0-indexed logic)
    alpha_2[k] = sum_var / (I - k - 1 - 1) # i.e. I - k - 2

# Extrapolate the final variance parameter (alpha_{I-1}^2)
# Using Mack's suggested heuristic to prevent instability
alpha_2[I-2] = min((alpha_2[I-3]**2) / alpha_2[I-4], alpha_2[I-3])

print("Estimated Variance Parameters (alpha_k^2):")
print(np.round(alpha_2, 2))

Estimated Variance Parameters (alpha_k^2):
[2.788348e+04 1.108530e+03 6.914400e+02 6.123000e+01 1.194400e+02
 4.082000e+01 1.340000e+00 7.880000e+00 7.880000e+00]


### Step 3: Calculate Standard Errors
Mack establishes closed-form solutions for the standard errors without assuming a specific underlying distribution.
The squared standard error for an individual accident year is:
$(s.e.(R_i))^2 = \hat{C}_{i,I}^2 \sum_{k=I+1-i}^{I-1} \frac{\alpha_k^2}{f_k^2} \left( \frac{1}{\hat{C}_{i,k}} + \frac{1}{\sum_{j=1}^{I-k} C_{j,k}} \right)$

The overall reserve standard error accounts for the correlations between different accident years generated by using the same development factors.

In [ ]:
# ==========================================
# 4. CALCULATE STANDARD ERRORS (MACK'S FORMULAS)
# ==========================================

# 1. Fix the final variance parameter extrapolation using Mack's Formula 9
alpha_2[I-2] = min((alpha_2[I-3]**2) / alpha_2[I-4], min(alpha_2[I-4], alpha_2[I-3]))

se_R = np.zeros(I)

# 2. Individual Standard Errors (Formula 7)
for i in range(1, I):
    var_sum = 0
    # Loop ONLY over the remaining future development periods for accident year i
    for k in range(I - 1 - i, I - 1):
        # Sum of known past claims in period k (the denominator in Mack's weights)
        sum_C_jk = np.nansum(C[:I - 1 - k, k])

        term1 = 1.0 / C_full[i, k]
        term2 = 1.0 / sum_C_jk

        var_sum += (alpha_2[k] / (f_k[k]**2)) * (term1 + term2)

    # Standard Error is the squared ultimate claims * the variance sum
    se_R[i] = C_full[i, I-1] * np.sqrt(var_sum)

# 3. Overall Standard Error incorporating Covariance (Formula 11)
var_overall = np.sum(se_R**2)
cov_sum = 0

# Double summation for every pair of accident years (i and j where i < j)
for i in range(1, I):
    for j in range(i + 1, I):
        term_sum = 0
        # The correlation only applies over the overlapping future periods
        for k in range(I - 1 - i, I - 1):
            sum_C_jk = np.nansum(C[:I - 1 - k, k])
            # Notice the covariance term lacks the 1/C_ik component!
            term_sum += (alpha_2[k] / (f_k[k]**2)) / sum_C_jk

        cov_sum += C_full[i, I-1] * C_full[j, I-1] * term_sum

var_overall += 2 * cov_sum
se_overall = np.sqrt(var_overall)

# ==========================================
# DISPLAY RESULTS
# ==========================================
# Use np.errstate to safely ignore the divide-by-zero warning just for this calculation
with np.errstate(divide='ignore', invalid='ignore'):
    cv_array = np.where(reserves > 0, (se_R / reserves) * 100, 0)

results_df = pd.DataFrame({
    "Reserve": reserves,
    "Std Error": se_R,
    "CV (%)": cv_array
}, index=[f"Acc_Yr_{i+1}" for i in range(I)])

print("Reserve Estimates and Variability by Accident Year:")
display(results_df.round(1))

print(f"\nOverall Reserve: {total_reserve:,.0f}")
print(f"Overall Standard Error: {se_overall:,.0f} (CV: {(se_overall/total_reserve)*100:.1f}%)")


Reserve Estimates and Variability by Accident Year:


,Reserve,Std Error,CV (%)
Acc_Yr_1,0.0,0.0,0.0
Acc_Yr_2,154.0,206.2,133.9
Acc_Yr_3,617.4,623.4,101.0
Acc_Yr_4,1636.1,747.2,45.7
Acc_Yr_5,2746.7,1469.5,53.5
Acc_Yr_6,3649.1,2001.9,54.9
Acc_Yr_7,5435.3,2209.2,40.6
Acc_Yr_8,10907.2,5357.9,49.1
Acc_Yr_9,10650.0,6333.2,59.5
Acc_Yr_10,16339.4,24566.3,150.3



Overall Reserve: 52,135
Overall Standard Error: 26,909 (CV: 51.6%)


### Step 4: Testing Assumption 2 (Calendar Year Effects)
A core chain ladder assumption is that accident years are independent. This is violated if there are calendar year effects (e.g., a change in claims handling speed or a surge in inflation affecting a specific calendar year diagonal).

Mack proposes a "Distribution-Free" test for this:
1. For each column $k$, calculate the median development factor.
2. Label each observed factor as "Larger" (L) or "Smaller" (S) than the median.
3. For each calendar year diagonal, count the number of L's and S's.
4. If a diagonal has a significant preponderance of L's or S's (measured via a binomial distribution probability), it suggests a calendar year distortion.

In [ ]:
# ==========================================
# 5. DIAGNOSTIC TEST: "LARGE/SMALL" CALENDAR YEAR EFFECT
# ==========================================

# Create an array to hold the "L", "S", or "*" (median/missing) labels
LS_matrix = np.full((I-1, I-1), "*", dtype=object)

# Assign L and S based on column medians
for k in range(I - 1):
    # Extract known factors for column k
    factors = []
    for i in range(I - k - 1):
        factors.append(C[i, k+1] / C[i, k])

    if len(factors) > 0:
        median_f = np.median(factors)
        for i in range(I - k - 1):
            val = C[i, k+1] / C[i, k]
            if val > median_f:
                LS_matrix[i, k] = "L"
            elif val < median_f:
                LS_matrix[i, k] = "S"
            else:
                LS_matrix[i, k] = "*" # Median itself is discarded in odd-numbered samples

print("L/S Matrix by Accident Year (Rows) and Development Year (Cols):")
display(pd.DataFrame(LS_matrix))

# Count L's and S's along Calendar Year Diagonals (j = i + k)
# We test j from 1 to I-2 (leaving out the very tips)
calendar_years = {}
for j in range(1, I - 1):
    L_count = 0
    S_count = 0

    for i in range(I - 1):
        for k in range(I - 1):
            if i + k == j:
                if LS_matrix[i, k] == "L":
                    L_count += 1
                elif LS_matrix[i, k] == "S":
                    S_count += 1

    n_total = L_count + S_count
    z_j = min(L_count, S_count)

    # Calculate Binomial Cumulative Probability: P(Z <= z_j) given p=0.5
    if n_total > 0:
        prob = stats.binom.cdf(z_j, n_total, 0.5) * 2 # Two-tailed test roughly
        calendar_years[f"Cal_Yr_{j}"] = {"L": L_count, "S": S_count, "n": n_total, "Z": z_j, "Prob(Z<=z)": prob}

print("\nCalendar Year Diagnostics:")
cal_df = pd.DataFrame(calendar_years).T
display(cal_df)

print("\nInterpretation:")
print("If 'Prob(Z<=z)' is very small (e.g., < 0.10), it indicates that either 'L' or 'S' factors dominate this calendar year.")
print("If such a trend exists, Mack suggests reducing the weight of these specific factors in the chain ladder calculation.")

L/S Matrix by Accident Year (Rows) and Development Year (Cols):


,0,1,2,3,4,5,6,7,8
0,S,S,S,S,L,L,*,S,*
1,L,S,L,L,*,S,L,L,*
2,S,S,*,S,L,S,S,*,*
3,S,S,L,S,S,L,*,*,*
4,L,L,L,L,S,*,*,*,*
5,*,L,S,L,*,*,*,*,*
6,L,L,S,*,*,*,*,*,*
7,L,L,*,*,*,*,*,*,*
8,S,*,*,*,*,*,*,*,*



Calendar Year Diagnostics:


,L,S,n,Z,Prob(Z<=z)
Cal_Yr_1,1.0,1.0,2.0,1.0,1.500000
Cal_Yr_2,0.0,3.0,3.0,0.0,0.250000
Cal_Yr_3,1.0,3.0,4.0,1.0,0.625000
Cal_Yr_4,3.0,1.0,4.0,1.0,0.625000
Cal_Yr_5,3.0,1.0,4.0,1.0,0.625000
Cal_Yr_6,4.0,2.0,6.0,2.0,0.687500
Cal_Yr_7,4.0,4.0,8.0,4.0,1.273438
Cal_Yr_8,4.0,4.0,8.0,4.0,1.273438



Interpretation:
If 'Prob(Z<=z)' is very small (e.g., < 0.10), it indicates that either 'L' or 'S' factors dominate this calendar year.
If such a trend exists, Mack suggests reducing the weight of these specific factors in the chain ladder calculation.


In [ ]:
# ==========================================
# 5. CALCULATE LOGNORMAL CONFIDENCE INTERVALS
# ==========================================

# 1. Calculate Lognormal parameters for the Overall Reserve (Mack's Formula 10)
# sigma^2 = ln(1 + (SE / R)^2)
cv_overall = se_overall / total_reserve
sigma_2_overall = np.log(1 + cv_overall**2)
sigma_overall = np.sqrt(sigma_2_overall)

# mu = ln(R) - sigma^2 / 2
mu_overall = np.log(total_reserve) - (sigma_2_overall / 2.0)

# 2. Calculate the 80% Confidence Interval (10th and 90th percentiles)
# Using scipy to find the exact Z-scores for the 10% and 90% boundaries
z_90 = stats.norm.ppf(0.90) # Approx 1.28
z_10 = stats.norm.ppf(0.10) # Approx -1.28

# Calculate Lognormal boundaries: exp(mu + Z * sigma)
upper_limit_80 = np.exp(mu_overall + z_90 * sigma_overall)
lower_limit_80 = np.exp(mu_overall + z_10 * sigma_overall)

print("Overall Reserve Confidence Intervals (Lognormal Approximation):")
print(f"Point Estimate:      {total_reserve:,.0f}")
print(f"80% Confidence Band: ({lower_limit_80:,.0f}, {upper_limit_80:,.0f})")

Overall Reserve Confidence Intervals (Lognormal Approximation):
Point Estimate:      52,135
80% Confidence Band: (24,852, 86,363)
